# ST-GCN Penn Action — Colab Entry Point

This notebook imports from the `penn-action-stgcn` repository.
All model, data, and training code lives in `src/` — this notebook
is for running experiments and visualisation only.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/penn-action-stgcn'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/<your-username>/penn-action-stgcn.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull

!pip install -q wandb pyyaml

import sys
sys.path.insert(0, REPO_DIR)

!tar -xf "/content/drive/MyDrive/Penn_Action.tar.gz" -C "/content/" 2>/dev/null || true

In [ ]:
# ── Preprocess (run once) ─────────────────────────────────────────────────────
from src.data.preprocess import preprocess_and_save

preprocess_and_save(
    mat_directory='/content/Penn_Action/labels',
    output_root='/content/Penn_Action/processed',
)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
import yaml
from src.training.train import run_train_eval

with open(f'{REPO_DIR}/configs/best_config.yaml') as f:
    cfg = yaml.safe_load(f)

# Override for this run (or edit configs/best_config.yaml directly)
cfg['adaptive'] = False
cfg['augmentation'] = True

test_acc, cm = run_train_eval(
    cfg,
    data_root='/content/Penn_Action/processed',
    model_save_path='/content/best_model.pth',
)
print(f'Test accuracy: {test_acc:.2f}%')

In [ ]:
# ── Evaluate a saved checkpoint ──────────────────────────────────────────────
from src.evaluation.evaluate import evaluate_model, plot_confusion_matrix

test_acc, cm = evaluate_model(
    checkpoint_path='/content/best_model.pth',
    test_data_path='/content/Penn_Action/processed/joint/test_data.npy',
    test_label_path='/content/Penn_Action/processed/joint/test_label.npy',
)

plot_confusion_matrix(cm, title=f'Test Set (acc: {test_acc:.2f}%)',
                      save_path='/content/confusion_matrix.png')